# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )
else:
    # Find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Working dir:", os.getcwd())
print("Dataset loaded:", df.shape)

Working dir: /content/flyrank-ml-internship-starter
Dataset loaded: (30000, 44)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane as an ML task

My lane is Refresh / Content Opportunity Scoring. I frame this primarily as a ranking problem because the goal is not simply to classify every page as good or bad. The goal is to rank pages so that the pages most deserving of human review appear near the top of the queue. This ranking can support decisions about which pages to review first for possible refresh, expansion, protection, pruning, or monitoring.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Task type: Ranking")
print("Unit of analysis: One page")
print("Decision: Which pages should be reviewed first?")

Task type: Ranking
Unit of analysis: One page
Decision: Which pages should be reviewed first?


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

For the starter version of this task, I would use **declining performance** as an observed proxy for identifying pages that may deserve review. The starter pipeline defines the label as whether `trend_direction` is `"down"`.

This is an observed outcome in the starter data rather than a claim that a page definitely needs a particular content action. A declining page may still require human review before deciding whether to refresh, expand, protect, prune, or monitor it.

I would not use `trend_direction` or `trend_pct` as model features because they are directly related to the label and could create target leakage.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print("\nShare declining:")
print(df["is_declining_label"].mean().round(3))

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Share declining:
0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric

My main success metric is Precision@50.

The reason is that the practical output is a prioritized review queue. Precision@50 measures how useful the first 50 recommendations are: among the 50 pages placed at the top of the queue, how many are actually declining according to the observed label.

A higher Precision@50 would mean that the limited human review effort is concentrated more heavily on pages that match the target outcome. I will compare the ML approach against a simple baseline rather than judging the model in isolation.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Primary success metric: Precision@50")
print("Meaning: fraction of the top 50 ranked pages that are actually declining.")

Primary success metric: Precision@50
Meaning: fraction of the top 50 ranked pages that are actually declining.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis

The unit of analysis is one page.

Each row represents one anonymized page and contains signals describing its search performance, content characteristics, and trend. The model would therefore produce a score or ranking for individual pages, which can then be turned into a review queue.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

page_slice = df[
    [
        "trend_direction",
        "search_volume",
        "impressions_90d",
        "ctr",
        "avg_position",
        "word_count",
        "content_age_days",
        "is_declining_label"
    ]
].copy()

print("One row = one page")
print("Rows:", len(page_slice))
print("Columns:", len(page_slice.columns))

display(page_slice.head(10))

One row = one page
Rows: 30000
Columns: 8


,trend_direction,search_volume,impressions_90d,ctr,avg_position,word_count,content_age_days,is_declining_label
0,down,10.0,3803,0.76,10.6,3221.0,187,1
1,down,90.0,15320,0.05,20.3,2481.0,445,1
2,down,0.0,12581,0.09,36.5,3515.0,141,1
3,stable,10.0,11751,0.49,6.2,NaN,463,0
4,down,0.0,19140,0.13,44.0,2803.0,263,1
5,down,720.0,3970,0.03,8.5,3080.0,147,1
6,down,0.0,20,0.00,7.0,3059.0,90,1
7,stable,590.0,1724,0.06,21.2,NaN,445,0
8,down,0.0,32574,0.09,46.0,3807.0,90,1
9,down,0.0,1240,0.16,4.9,NaN,257,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule here

A fixed rule can be useful as a transparent baseline, but page performance depends on multiple signals rather than one simple condition. Search volume, impressions, CTR, position, content characteristics, and other variables can interact in different ways.

An ML model can learn patterns across multiple features and produce a ranked set of pages instead of relying on one manually chosen threshold. This makes it possible to test whether the learned approach provides better prioritization than a fixed rule using a measurable metric such as Precision@50.

The model would still be decision-support. A high-ranked page would be a candidate for human review, not an automatic instruction to change the page.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features_to_check = [
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "content_age_days"
]

display(df[features_to_check].describe().T)

,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
word_count,22301.0,3107.760325,1452.382598,8.0,2413.0,2877.00,3666.00,9546.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.